In [ ]:
# Imports

## Core libraries
import numpy as np
import matplotlib.pyplot as plt

## TensorFlow / Keras
import tensorflow as tf
import keras
from keras import ops, layers
from keras.callbacks import Callback
from keras.models import Sequential, Model
from keras.layers import Input, Conv2D, Dense


In [ ]:
# Setting seeds

np.random.seed(1999)
tf.random.set_seed(1999)


In [ ]:
# Loading the dataset

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

## Feature scaling (mapping pixel values to [0,1])
'''
This is used as neural network training via gradient descent is sensitive to the
scale of inputs (through activations a = Wx + b). This causes gradients to vanish
and makes the loss landscape poorly conditioned.
'''
x_train_full = x_train_full / 255.0
x_test = x_test / 255.0

''' Recalling our y values are our categories '''
y_train_full = keras.utils.to_categorical(y_train_full, 10)
y_test = keras.utils.to_categorical(y_test, 10)

## Setting up our validation dataset
n_val = int(0.2 * len(x_train_full))
perm = np.random.permutation(len(x_train_full))
val_idx, train_idx = perm[:n_val], perm[n_val:]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]
x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]

## Check
print(f"train: {x_train.shape} val: {x_val.shape} test: {x_test.shape}")


In [ ]:
# Augmenting the dataset
'''
This involves creating new training examples by applying label preserving
transformations to existing examples e.g. flips, crops, scaling, colour jitter,
adding noise, ...
'''

'''
Applying three transformations in sequence, we pad the image with black 0s,
then randomly crop the newly padded image (so the model learns that a shifted
horse picture is still a horse), and then randomly mirroring the image from left
to right.
'''
augment = keras.Sequential([layers.ZeroPadding2D(padding=4),
                            layers.RandomCrop(32, 32),
                            layers.RandomFlip("horizontal")])

batch_size = 128

'''
This builds the full training data pipeline, where:
- The from_tensor_slices((x_train, y_train)) wraps the numpy arrays into a
  tf.data.Dataset that has image label pairs one at a time.
- For .shuffle(len(x_train)) - this shifts the entire dataset after each epoch.
- For .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
  this applies the augmentation pipeline to each batch of images whilst leaving
  labels untouched. The training=True is important because RandomCrop and
  RandomFlip only apply their transformations during training. The
  num_parallel_calls=tf.data.AUTOTUNE lets TensorFlow decide how many batches to
  augment in parallel across CPU threads.
'''
train_ds = (tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(len(x_train)).batch(batch_size).map(lambda x, y: (augment(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE))

### Recalling that there is no augmentation for validation and test datasets
val_ds  = (tf.data.Dataset.from_tensor_slices((x_val,  y_val ))
           .batch(batch_size).prefetch(tf.data.AUTOTUNE))
test_ds = (tf.data.Dataset.from_tensor_slices((x_test, y_test))
           .batch(batch_size).prefetch(tf.data.AUTOTUNE))


In [ ]:
# Residual block (no dropout)
'''
Here is the specification requirements for a single residual block (the repeating
unit in ResNet-20). Instead of learning the mapping between the input and the
output, we instead learn the difference between the input and the desired output
(the residual).

In this, the shortcut path passes the input unchanged (albeit with a channel
count change to match shapes at times). These two paths are added together for
a final ReLU.
'''

### We recall that a convolution slides a small window (a kernel/filter) across our grid.
### A single kernel produces one output grid (one channel). One kernel might learn to detect vertical edges, another colour gradients, and so on.
### We thus use 16 kernels simultaneously to produce an output with 16 channels.

class ResidualBlock(keras.layers.Layer):

    '''
    The init method constructs everything which doesn't fit into the other categories.
    In this, we construct the two convolutional layers and the ReLU activation.
    Everything here is created once and reused when the block processes an input.
    Projection is handled separately, as the input's channel count changes.
    '''
    def __init__(self, filters, stride=1, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.stride = stride

        ### The same padding here adds just enough padding at each convolution step so the output spatial dimensions match the input.
        ### The kernel initialiser establishes starting weights for the kernels such that they are not all zero (each kernel learns the same thing), too large (where activations explode), or too small (vanishing to zero).
        self.conv1 = layers.Conv2D(filters, 3, strides=stride, padding="same",
                                   kernel_regularizer=keras.regularizers.L2(1e-4),
                                   kernel_initializer="he_normal")

        ### Batch normalisation ensures each channel has a mean of 0 and a variance of 1.
        ### This is necessary for faster and more stable training as the weights update during training, causing our outputs to shift considerably (and calibrated to the old weights).
        ### The purpose of batch normalisation is to ensure a stable range of outputs.
        self.bn1 = layers.BatchNormalization()
        self.conv2 = layers.Conv2D(filters, 3, padding="same",
                                   kernel_regularizer=keras.regularizers.L2(1e-4),
                                   kernel_initializer="he_normal")
        self.bn2 = layers.BatchNormalization()
        self.relu = layers.Activation("relu")

    ### Going deeper into the network, we want to detect increasingly abstract features.
    ### We therefore downsample (have a bigger stride) as we detect combinations.
    ### We compensate for that by increasing the number of channels.

    '''
    The build method handles the projection shortcut (and handles shape-matching if needed).
    '''
    def build(self, input_shape):
        in_channels = input_shape[-1]

        if self.stride != 1 or in_channels != self.filters:
            self.proj = layers.Conv2D(self.filters, 1, strides=self.stride, padding="valid",
                                      kernel_regularizer=keras.regularizers.L2(1e-4),
                                      kernel_initializer="he_normal")
            self.bn_proj = layers.BatchNormalization()
        else:
            self.proj = None

        super().build(input_shape)

    '''
    The call method runs the actual forward pass of the model organising the
    split into the main convolutional path and the shortcut path, then summing
    them and applying ReLU.
    '''
    def call(self, x):
        if self.proj is not None:
            shortcut = self.bn_proj(self.proj(x))
        else:
            shortcut = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        return self.relu(out + shortcut)

    '''
    The get_config method enables serialisation so Keras can save/load the model.
    '''
    def get_config(self):
        config = super().get_config()
        config.update({"filters": self.filters,
                       "stride": self.stride})
        return config


In [ ]:
# build_resnet20_vib — VIB alone variant (no SWAG / no weight uncertainty)
'''
The feature extractor follows the deterministic ResNet-20 specification above.
The only architectural change is after global average pooling: instead of sending
the feature vector directly to the classifier, we map it to the parameters of a
diagonal Gaussian bottleneck, sample z, and classify from z.

This is VIB alone. The weights are fixed after training; uncertainty at test time
comes only from repeatedly sampling the bottleneck representation z.
'''

class VIBSampling(keras.layers.Layer):

    """
    Reparameterisation layer for VIB.

    Given mu and log_var, sample z = mu + sigma * epsilon and add the VIB KL
    penalty beta * KL(q(z|x) || N(0, I)) to the model loss.
    """
    def __init__(self, beta=1e-3, **kwargs):
        super().__init__(**kwargs)
        self.beta = beta

    def call(self, inputs):
        mu, log_var = inputs
        eps = tf.random.normal(shape=tf.shape(mu), dtype=mu.dtype)
        z = mu + tf.exp(0.5 * log_var) * eps

        kl_per_example = -0.5 * tf.reduce_sum(
            1.0 + log_var - tf.square(mu) - tf.exp(log_var), axis=-1
        )
        self.add_loss(self.beta * tf.reduce_mean(kl_per_example))

        return z

    def get_config(self):
        config = super().get_config()
        config.update({"beta": self.beta})
        return config


def build_vib_resnet20(beta=1e-3, latent_dim=64):
    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Conv2D(16, 3, padding="same",
                      kernel_regularizer=keras.regularizers.L2(1e-4),
                      kernel_initializer="he_normal")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    for _ in range(3):
        x = ResidualBlock(16)(x)

    x = ResidualBlock(32, stride=2)(x)
    for _ in range(2):
        x = ResidualBlock(32)(x)

    x = ResidualBlock(64, stride=2)(x)
    for _ in range(2):
        x = ResidualBlock(64)(x)

    ### Global average pooling takes the block's output and turns it into a flat
    ### feature vector. In the deterministic version this went straight into the
    ### final dense layer; here it becomes the input to the VIB bottleneck.
    x = layers.GlobalAveragePooling2D()(x)

    z_mu = layers.Dense(latent_dim,
                        kernel_initializer="he_normal",
                        name="z_mu")(x)
    z_log_var = layers.Dense(latent_dim,
                             kernel_initializer="zeros",
                             bias_initializer="zeros",
                             name="z_log_var")(x)
    z = VIBSampling(beta=beta, name="vib_bottleneck")([z_mu, z_log_var])

    outputs = layers.Dense(10, kernel_initializer="he_normal")(z)

    return keras.Model(inputs, outputs)


In [ ]:
# β sweep — configuration
'''
§4.3 of the proposal specifies β ∈ {10⁻⁴, 10⁻³, 10⁻², 10⁻¹, 1} and selecting
the value that best balances accuracy and calibration on the held-out validation
set. This cell defines the sweep; cell 8 runs it.

Selection criterion: MC-averaged validation NLL (cross-entropy only, no KL term).
Using raw val_loss would be misleading because a larger β adds a proportionally
larger KL penalty, making the val_loss values incomparable across β values.
MC-averaged NLL is a pure measure of predictive quality that jointly captures
accuracy and calibration, consistent with §4.3.

M_SWEEP controls how many bottleneck samples are drawn per input when ranking
candidates. Five is sufficient for selection; the final model uses M=30 as
specified in the proposal.
'''
BETA_VALUES = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
latent_dim  = 64   # equal to the ResNet-20 pooled feature width
M_SWEEP     = 5    # MC samples for val evaluation during sweep

beta_sweep_results = {}   # beta -> {val_acc, val_nll, checkpoint, history}


In [ ]:
# Training VIB-alone model

'''
Step-wise learning-rate schedule: drop by 10x at epochs 100 and 150.
This is unchanged from the deterministic notebook so that the VIB-alone model
uses the same optimisation recipe.
'''
def lr_schedule(epoch, lr):
    if epoch in (100, 150):
        return lr * 0.1
    return lr


In [ ]:
# β sweep — training and validation evaluation
'''
One full 200-epoch run per β value using the same SGD schedule as every other
baseline. At the end of each run the best checkpoint is loaded and we draw
M_SWEEP=5 stochastic bottleneck samples per validation input to compute:
  val_acc  — MC-averaged accuracy
  val_nll  — MC-averaged NLL (pure cross-entropy; no KL term)

Only lightweight Python objects (scalars, checkpoint paths, history dicts) are
accumulated across iterations; the Keras model itself is not. This means
clear_session() at the top of each iteration is safe — there are no live model
references in beta_sweep_results to corrupt.
'''

val_labels = np.argmax(y_val, axis=1)   # integer labels for val evaluation

for beta_val in BETA_VALUES:
    print("\n" + "=" * 60)
    print(f"Training VIB-alone  β = {beta_val:.0e}")
    print("=" * 60)

    keras.backend.clear_session()   # safe: no live models in beta_sweep_results
    np.random.seed(1999)
    tf.random.set_seed(1999)

    m = build_vib_resnet20(beta=beta_val, latent_dim=latent_dim)
    m.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1, momentum=0.9),
              loss=keras.losses.CategoricalCrossentropy(from_logits=True),
              metrics=["categorical_accuracy"])

    ckpt_path = f"vib_resnet20_beta{beta_val:.0e}_best.weights.h5"
    sweep_callbacks = [
        keras.callbacks.ModelCheckpoint(ckpt_path,
                                        monitor="val_loss",
                                        save_best_only=True,
                                        save_weights_only=True),
        keras.callbacks.LearningRateScheduler(lr_schedule, verbose=0)
    ]

    hist = m.fit(train_ds, epochs=200, validation_data=val_ds,
                 callbacks=sweep_callbacks, verbose=1)
    m.load_weights(ckpt_path)

    # MC-averaged val NLL and accuracy — KL-free, comparable across β values
    val_logits_mc = np.stack([m.predict(val_ds, verbose=0)
                              for _ in range(M_SWEEP)], axis=0)  # (M_SWEEP, N_val, 10)
    val_probs_mc  = tf.nn.softmax(val_logits_mc, axis=-1).numpy().mean(axis=0)
    val_acc = (np.argmax(val_probs_mc, axis=1) == val_labels).mean()
    val_nll = -np.log(val_probs_mc[np.arange(len(val_labels)), val_labels] + 1e-12).mean()

    # Store scalars and the plain history dict — no live Keras objects
    beta_sweep_results[beta_val] = {
        "val_acc":    float(val_acc),
        "val_nll":    float(val_nll),
        "checkpoint": ckpt_path,
        "history":    hist.history   # plain dict of lists, survives clear_session()
    }
    print(f"  β = {beta_val:.0e} | val_acc = {val_acc:.4f} | val_nll = {val_nll:.4f}")

# Select best β: lowest MC-averaged validation NLL
best_beta = min(beta_sweep_results, key=lambda b: beta_sweep_results[b]["val_nll"])
print(f"\nBest β = {best_beta:.0e}  "
      f"(val_nll = {beta_sweep_results[best_beta]['val_nll']:.4f}  "
      f"val_acc = {beta_sweep_results[best_beta]['val_acc']:.4f})")


In [ ]:
# Reload best model from sweep
'''
Start a fresh Keras session and rebuild the best-β model from its checkpoint.
This is the model used for all downstream predictions and evaluation.
The single-sample test accuracy printed here is a sanity check; the MC-averaged
number in the predictions cell is the figure to report.
Note: test_loss includes the KL penalty term (via add_loss in VIBSampling)
and so should not be compared directly with the deterministic baseline's loss.
'''
keras.backend.clear_session()   # clean slate before rebuilding
beta = best_beta

model = build_vib_resnet20(beta=beta, latent_dim=latent_dim)
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1, momentum=0.9),
              loss=keras.losses.CategoricalCrossentropy(from_logits=True),
              metrics=["categorical_accuracy"])
model.load_weights(beta_sweep_results[best_beta]["checkpoint"])

test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Selected β = {beta:.0e}")
print(f"Single-sample test loss (includes KL): {test_loss:.4f}")
print(f"Single-sample test accuracy:           {test_acc:.4f}")


In [ ]:
# β sweep — results summary

print(f"{'β':>8}  {'val_acc':>10}  {'val_nll':>10}")
print("-" * 34)
for bv in BETA_VALUES:
    r = beta_sweep_results[bv]
    marker = "  ← selected" if bv == best_beta else ""
    print(f"{bv:>8.0e}  {r['val_acc']:>10.4f}  {r['val_nll']:>10.4f}{marker}")


In [ ]:
# Plotting β sweep overview and best-β training curves

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

## Panel 1: MC val NLL per β — sweep selection criterion
bv_labels = [f"{bv:.0e}" for bv in BETA_VALUES]
nll_vals   = [beta_sweep_results[bv]["val_nll"] for bv in BETA_VALUES]
colours    = ["tab:orange" if bv == best_beta else "steelblue" for bv in BETA_VALUES]
axes[0].bar(bv_labels, nll_vals, color=colours)
axes[0].set_xlabel("β"); axes[0].set_ylabel("MC val NLL")
axes[0].set_title("β sweep — validation NLL\n(orange = selected)")

## Panels 2 & 3: train / val loss and accuracy for the selected β
best_hist = beta_sweep_results[best_beta]["history"]

axes[1].plot(best_hist["loss"],     label="train loss")
axes[1].plot(best_hist["val_loss"], label="val loss")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("loss")
axes[1].set_title(f"VIB-alone β={best_beta:.0e} — Loss"); axes[1].legend()

axes[2].plot(best_hist["categorical_accuracy"],     label="train acc")
axes[2].plot(best_hist["val_categorical_accuracy"], label="val acc")
axes[2].set_xlabel("epoch"); axes[2].set_ylabel("accuracy")
axes[2].set_title(f"VIB-alone β={best_beta:.0e} — Accuracy"); axes[2].legend()

plt.tight_layout(); plt.show()


In [ ]:
# Loading SVHN as the out-of-distribution reference set
'''
Per the proposal §3.5, SVHN (street view of house numbers) is used as the
OOD test set. SVHN images are 32x32 colour photos like CIFAR-10, but of
digits, so a CIFAR-10 trained model should ideally report higher predictive
entropy on SVHN inputs than on its in-distribution CIFAR-10 test images.

The whole SVHN test split is ~26k examples; we load it via
tensorflow_datasets and rescale to [0, 1] to match CIFAR-10 preprocessing.
'''
import tensorflow_datasets as tfds

svhn = tfds.load("svhn_cropped", split="test", as_supervised=True)
svhn_images = []
for img, _ in svhn:
    svhn_images.append(img.numpy())
x_svhn = np.stack(svhn_images).astype("float32") / 255.0

print(f"SVHN OOD set: {x_svhn.shape}")


In [ ]:
# Loading CIFAR-10-C as the second out-of-distribution reference set
'''
Per the proposal §3.5, CIFAR-10-C (Hendrycks & Dietterich 2019) is the
second OOD dataset. Unlike SVHN which represents a full domain shift,
CIFAR-10-C tests calibration under in-distribution corruption: blur, noise,
weather effects, digital artefacts, etc. This is a distinct failure mode
from domain shift and is evaluated separately.

We load all 15 standard corruption types at severity 3 (mid-level) and
concatenate them into a single flat evaluation array. Labels are the
original CIFAR-10 test labels (repeated once per corruption type), so we
can also report accuracy under corruption alongside OOD confidence.

TFDS name format: "cifar10_corrupted/{corruption_type}_{severity}"
'''

# 15 standard corruptions from Hendrycks & Dietterich (2019)
CORRUPTIONS = [
    "brightness", "contrast", "defocus_blur", "elastic_transform",
    "fog", "frost", "glass_blur", "gaussian_noise",
    "impulse_noise", "jpeg_compression", "motion_blur",
    "pixelate", "shot_noise", "snow", "zoom_blur"
]
SEVERITY = 3  # severity levels 1–5; 3 is the standard mid-level benchmark

cifar10c_images = []
cifar10c_labels = []
loaded_corruptions = []

for corruption in CORRUPTIONS:
    ds_name = f"cifar10_corrupted/{corruption}_{SEVERITY}"
    try:
        ds = tfds.load(ds_name, split="test", as_supervised=True)
        for img, label in ds:
            cifar10c_images.append(img.numpy())
            cifar10c_labels.append(label.numpy())
        loaded_corruptions.append(corruption)
    except Exception as e:
        print(f"  Warning — skipping {corruption}: {e}")

x_cifar10c = np.stack(cifar10c_images).astype("float32") / 255.0
y_cifar10c = np.array(cifar10c_labels)   # integer labels 0–9

print(f"CIFAR-10-C OOD set: {x_cifar10c.shape}")
print(f"  Severity: {SEVERITY}  |  Corruptions loaded: {len(loaded_corruptions)}/{len(CORRUPTIONS)}")


In [ ]:
# Generating predictions for the evaluation file
'''
This is the only output the downstream evaluation notebook needs from this
file. We produce softmax probabilities on:
  - the CIFAR-10 test set (in-distribution),
  - the SVHN test set (out-of-distribution, full domain shift),
  - the CIFAR-10-C test set (out-of-distribution, in-distribution corruption).

The VIB-alone model has stochasticity at the representation level only. The MC
axis is repeated bottleneck samples with fixed trained weights — there is no
weight-level uncertainty in this notebook.
'''

def predict_vib_mc(model, data, mc_samples=30, batch_size=128):
    all_probs = []

    for s in range(mc_samples):
        if isinstance(data, tf.data.Dataset):
            logits = model.predict(data, verbose=0)
        else:
            logits = model.predict(data, batch_size=batch_size, verbose=0)

        probs = tf.nn.softmax(logits, axis=-1).numpy()
        all_probs.append(probs)

        print(f"MC sample {s + 1:02d}/{mc_samples} complete")

    all_probs  = np.stack(all_probs, axis=0)   # (M, N, 10)
    mean_probs = all_probs.mean(axis=0)         # (N, 10)
    return mean_probs, all_probs

n_mc_samples = 30

## In-distribution: CIFAR-10 test set
print("Running MC inference on CIFAR-10 test set...")
probs_test, all_probs_test = predict_vib_mc(model, test_ds, mc_samples=n_mc_samples)
preds_test  = np.argmax(probs_test, axis=1)
labels_test = np.argmax(y_test, axis=1)
print(f"MC-averaged test accuracy: {(preds_test == labels_test).mean():.4f}")

## Out-of-distribution: SVHN (full domain shift)
print("Running MC inference on SVHN OOD set...")
probs_svhn, all_probs_svhn = predict_vib_mc(model, x_svhn,
                                             mc_samples=n_mc_samples, batch_size=128)

## Out-of-distribution: CIFAR-10-C (in-distribution corruption)
print("Running MC inference on CIFAR-10-C OOD set...")
probs_cifar10c, all_probs_cifar10c = predict_vib_mc(model, x_cifar10c,
                                                      mc_samples=n_mc_samples, batch_size=128)
preds_cifar10c = np.argmax(probs_cifar10c, axis=1)
print(f"Accuracy under corruption (severity {SEVERITY}): "
      f"{(preds_cifar10c == y_cifar10c).mean():.4f}")

print(f"\nall_probs_test:     {all_probs_test.shape}")
print(f"all_probs_svhn:     {all_probs_svhn.shape}")
print(f"all_probs_cifar10c: {all_probs_cifar10c.shape}")


In [ ]:
# Saving raw outputs for the evaluation file
'''
Save format (shared across all benchmarks). The evaluation notebook will
load these by filename and produce ECE, reliability diagrams, OOD AUROC,
risk-coverage and uncertainty summaries.

Keys:
  model_name            - human-readable string identifier
  beta                  - selected VIB compression strength
  latent_dim            - bottleneck dimensionality
  n_mc_samples          - number of representation samples at inference (30)
  probs_test            - (N_test, 10)      mean predictive dist. on CIFAR-10 test
  labels_test           - (N_test,)         true labels (integer 0–9)
  preds_test            - (N_test,)         argmax of probs_test
  probs_svhn            - (N_svhn, 10)      mean predictive dist. on SVHN OOD
  probs_cifar10c        - (N_c10c, 10)      mean predictive dist. on CIFAR-10-C OOD
  labels_cifar10c       - (N_c10c,)         true labels for CIFAR-10-C (integer 0–9)
  preds_cifar10c        - (N_c10c,)         argmax of probs_cifar10c
  cifar10c_severity     - scalar int         corruption severity used (3)
  all_probs_test        - (M, N_test, 10)   per-sample probs (M=30 bottleneck samples)
  all_probs_svhn        - (M, N_svhn, 10)   per-sample probs
  all_probs_cifar10c    - (M, N_c10c, 10)   per-sample probs
  beta_sweep_betas      - (5,)              β values tried in sweep
  beta_sweep_val_nlls   - (5,)              MC val NLL for each β
  beta_sweep_val_accs   - (5,)              MC val accuracy for each β

There is intentionally no weight-sample axis. The MC axis is bottleneck /
representation stochasticity only.
'''
np.savez("vib_alone_results.npz",
         model_name="vib_alone",
         beta=beta,
         latent_dim=latent_dim,
         n_mc_samples=n_mc_samples,
         # In-distribution
         probs_test=probs_test,
         labels_test=labels_test,
         preds_test=preds_test,
         # OOD — full domain shift
         probs_svhn=probs_svhn,
         all_probs_svhn=all_probs_svhn,
         # OOD — in-distribution corruption
         probs_cifar10c=probs_cifar10c,
         labels_cifar10c=y_cifar10c,
         preds_cifar10c=preds_cifar10c,
         cifar10c_severity=SEVERITY,
         # Per-sample arrays (M = bottleneck samples, no weight axis)
         all_probs_test=all_probs_test,
         all_probs_cifar10c=all_probs_cifar10c,
         # β sweep provenance
         beta_sweep_betas=np.array(BETA_VALUES),
         beta_sweep_val_nlls=np.array([beta_sweep_results[b]["val_nll"] for b in BETA_VALUES]),
         beta_sweep_val_accs=np.array([beta_sweep_results[b]["val_acc"] for b in BETA_VALUES]))

print("Saved vib_alone_results.npz")
